# 🔍 Retrievers

## What is a Retriever?
A **Retriever** is an abstraction over vector stores (and other sources) that fetches relevant documents.

```python
retriever.invoke("my question") → List[Document]
```

## Why Retrievers?
Vector stores have many search methods. Retrievers:
- Provide a **uniform interface** (always returns `List[Document]`)
- Can use **multiple strategies** (not just vector similarity)
- Integrate directly with LCEL chains
- Work with ANY document source, not just vector stores

## Retrieval Strategies
| Strategy | How It Works | Best For |
|----------|-------------|---------|
| Vector similarity | Cosine similarity on embeddings | Semantic search |
| MMR | Max Marginal Relevance — diverse results | Avoiding redundancy |
| BM25 | Keyword-based search | Exact term matching |
| Hybrid | Vector + BM25 combined | Best of both worlds |
| Self-query | LLM generates filters | Structured data |
| Multi-query | LLM generates multiple queries | Better recall |


In [ ]:
# ── Basic Retriever from Vector Store ────────────────────────────────────
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

docs = [
    Document(page_content='The capital of France is Paris.'),
    Document(page_content='Python was created by Guido van Rossum.'),
    Document(page_content='LangChain was created by Harrison Chase.'),
    Document(page_content='FAISS is a library for efficient similarity search.'),
]

vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings(model='text-embedding-3-small'))

# Convert vector store to retriever
retriever = vectorstore.as_retriever(
    search_type='similarity',  # 'similarity', 'mmr', 'similarity_score_threshold'
    search_kwargs={'k': 2}     # return top 2 documents
)

# Use the retriever
results = retriever.invoke('Who created Python?')
print(f'Retriever type: {type(retriever).__name__}')
print(f'Results: {len(results)}')
for doc in results:
    print(f'  → {doc.page_content}')

In [ ]:
# ── MMR Retriever: Maximum Marginal Relevance ───────────────────────────
# WHY MMR? Standard similarity search can return redundant results.
# MMR balances: relevance to query + diversity of results.
# fetch_k docs → re-rank to maximize diversity → return k docs

mmr_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 2,        # return 2 docs
        'fetch_k': 4,  # consider top 4 candidates
        'lambda_mult': 0.7  # 1.0=max relevance, 0.0=max diversity
    }
)

results = mmr_retriever.invoke('Who created things?')
print('MMR Results (diverse):')
for doc in results:
    print(f'  → {doc.page_content}')

In [ ]:
# ── MultiQueryRetriever: LLM-Powered Retrieval ──────────────────────────
# WHY: A single query may miss relevant documents.
# MultiQueryRetriever generates MULTIPLE queries from one question,
# retrieves docs for each, then deduplicates results.

from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k': 2}),
    llm=llm
)

# Single user question
question = 'Tell me about who created popular tools'
results = multi_query_retriever.invoke(question)

print(f'Original question: {question}')
print(f'Docs retrieved: {len(results)}')
for doc in results:
    print(f'  → {doc.page_content}')

## 🎯 Interview Questions

1. **What is a Retriever in LangChain?**
2. **Difference between a vector store and a retriever?**
3. **What is MMR (Maximum Marginal Relevance)?**
4. **What problem does MultiQueryRetriever solve?**
5. **What search_type options does as_retriever() support?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Retrievers — Fetching Relevant Documents**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
